# Bulletin languages on Kaggle: fine-tune the translation model on two T4 GPUs

This notebook fine-tunes NLLB-200 from English into the languages of ICPAC's member states
on Kaggle's free **GPU T4 x2**, so the Copilot and the weekly bulletin can be given in them.
It runs the scripts in `training/translation` of the service's repository
(`prepare_data.py`, `train.py`, `evaluate.py`, `translate.py`); `docs/language_training.md`
has the plan, the datasets and their licences.

**Before running**, in the notebook's settings (right-hand panel):

1. *Accelerator*: **GPU T4 x2**. *Internet*: **on**. Kaggle shows the Internet switch only
   after the account's phone number is verified (kaggle.com → Settings → Phone
   verification). Without it, the first cell stops and says so.
2. *Add-ons → Secrets*: add **HF_TOKEN**, a Hugging Face read token. Public data works
   without it; the gated SALT sentences (Luganda, Acholi, Lugbara, Runyankole, Ateso) need
   it, after accepting SALT's terms on its Hugging Face page.
3. Run it with **Save Version → Save & Run All**, so it keeps running with the browser
   closed. Kaggle stops a GPU session after 9 or 12 hours (check your account's quota page;
   the weekly GPU quota is about 30 hours); training saves and stops itself before that.

**To continue in a new session**: open the notebook, *Add Input → Your Work → this
notebook's output*, and run it again. It finds the newest checkpoint and the prepared data
in the earlier output and carries on.

In [ ]:
# Settings: change these, then Save & Run All.
PHASE = "1"  # 1: Kiswahili, Amharic, Somali, Afaan Oromo; 2: Tigrinya, Kinyarwanda, Kirundi;
# 3: Luganda, Dholuo, Kikuyu, Kamba; 4: Afar, Tigre, Sudanese Arabic, Acholi, Alur, Rukiga,
# Lugbara, Runyankole, Ateso, Dinka, Nuer (little data: new language codes, review needed)
LANGUAGES = None  # or your own list, e.g. ["sw", "luo", "ki"]; None takes the phase's
MODEL = "facebook/nllb-200-distilled-600M"  # or "facebook/nllb-200-distilled-1.3B" with LORA
LORA = False  # True for the 1.3B model (two T4s have 16 GB each)
EPOCHS = 1  # one pass fits a session for phase 1; 2 once the first run looks good
BATCH = 8  # per GPU: 2 GPUs x 8 x 4 accumulation = 64 sentence pairs per update
ACCUMULATE = 4
MINED_ROWS = 50_000  # general sentence pairs per language, best-scoring first
HOURS = 8.5  # stop and save before Kaggle ends the session (9 or 12 hours)
EVALUATE_BASELINE = True  # score the untouched model first, to compare
PUSH_TO_HUB = None  # e.g. "your-name/icpac-nllb600m-phase1" (a private model), or None
REPO = "https://github.com/Joe254h/ICPAC_ML_AI_PRODUCT"
BRANCH = "main"
RUN = "/kaggle/working/runs/nllb"

## 1. Internet and GPUs

In [ ]:
import socket
import subprocess

import torch


def reachable(host: str) -> bool:
    try:
        socket.getaddrinfo(host, 443)
        return True
    except OSError:
        return False


offline = [h for h in ("pypi.org", "github.com", "huggingface.co") if not reachable(h)]
if offline:
    raise RuntimeError(
        f"This session has no internet (cannot reach {', '.join(offline)}). Turn on "
        "Internet in the notebook's settings (right-hand panel, Session options), then run "
        "it again. Kaggle shows that switch only after the account's phone number is "
        "verified: kaggle.com, Settings, Phone verification."
    )
print("Internet: on")


def check(step: str) -> None:
    # IPython keeps the exit status of the last ! command in _exit_code.
    if _exit_code:
        raise RuntimeError(f"{step} failed (exit status {_exit_code}): see the lines above")


GPUS = torch.cuda.device_count()
print(subprocess.run(["nvidia-smi", "-L"], capture_output=True, text=True).stdout)
assert GPUS >= 1, "Turn on a GPU accelerator (GPU T4 x2) in the notebook settings"
# A T4 (compute capability 7.5) has no bf16; PyTorch may still report it, as emulated.
BF16 = torch.cuda.get_device_capability(0)[0] >= 8
print(f"{GPUS} GPU(s) ({torch.cuda.get_device_name(0)}); precision: {'bf16' if BF16 else 'fp16'}")
if GPUS < 2:
    print("Only one GPU: training works but takes about twice as long")

## 2. Install the libraries (Kaggle already has PyTorch)

In [ ]:
import sys

!{sys.executable} -m pip install --quiet "transformers>=4.46,<5" "datasets>=3.0" "accelerate>=1.0" "peft>=0.13" sentencepiece sacrebleu
check("Installing the libraries")

## 3. Fetch the training scripts

In [ ]:
import os
import shutil

WORK = "/kaggle/working/translation"
if not os.path.isdir(WORK):
    shutil.rmtree("/tmp/icpac-repo", ignore_errors=True)
    !git clone --quiet --depth 1 --branch {BRANCH} --filter=blob:none --sparse {REPO} /tmp/icpac-repo
    check("Fetching the repository")
    !git -C /tmp/icpac-repo sparse-checkout set training/translation
    check("Fetching the training scripts")
    shutil.copytree("/tmp/icpac-repo/training/translation", WORK)
%cd {WORK}

sys.path.insert(0, WORK)
from languages import LANGUAGES as KNOWN
from languages import PHASES

CHOSEN = LANGUAGES or PHASES[PHASE]
print("Training:", ", ".join(f"{KNOWN[k].name} ({k})" for k in CHOSEN))
new = [KNOWN[k].name for k in CHOSEN if KNOWN[k].init_from]
if new:
    print("New to NLLB (started from a related language):", ", ".join(new))

## 4. Sign in to Hugging Face (for the gated SALT sentences)

In [ ]:
# Phase 1 does not need this: only the SALT sentences (most needed by Lugbara, Runyankole
# and Ateso in phases 3 and 4) are gated.
token = None
try:
    from kaggle_secrets import UserSecretsClient

    token = UserSecretsClient().get_secret("HF_TOKEN")
except Exception as exc:
    print(
        "Kaggle gave this run no secret called HF_TOKEN. In Add-ons > Secrets, the label "
        "must be exactly HF_TOKEN and its switch must be on for THIS notebook (an imported "
        "copy is a new notebook); then Save Version again, since a running version keeps "
        f"the secrets it started with. Kaggle said: {type(exc).__name__}: {exc}"
    )
if token:
    from huggingface_hub import login, whoami

    try:
        login(token=token.strip())
        print("Signed in to Hugging Face as", whoami()["name"])
    except Exception as exc:
        print(f"Hugging Face refused the token ({type(exc).__name__}): create a new read token")
if not token:
    print("SALT is skipped; everything else works")

## 5. Continue from an earlier session, if one is attached

In [ ]:
import glob
import re

RESUME = None
earlier = sorted(
    glob.glob("/kaggle/input/**/runs/nllb/checkpoint-*", recursive=True),
    key=lambda path: int(re.search(r"checkpoint-(\d+)", path).group(1)),
)
if earlier:
    RESUME = earlier[-1]
    print("Continuing from", RESUME)
held = glob.glob("/kaggle/input/**/translation/data", recursive=True)
if held and not os.path.isdir("data"):
    # The same prepared data, so the resumed run sees the sentences in the same order.
    shutil.copytree(held[0], "data")
    print("Prepared data copied from", held[0])

## 6. Prepare the data (about 5-15 minutes a language: downloads, then filtering)

In [ ]:
missing = [k for k in CHOSEN if not os.path.isfile(f"data/{k}/train.jsonl")]
if missing:
    !python prepare_data.py --languages {" ".join(missing)} --out data --templates translations --mined-rows {MINED_ROWS}
    check("Preparing the data")
!wc -l data/*/train.jsonl data/*/test.jsonl

## 7. Score the untouched model (the baseline to beat)

In [ ]:
if EVALUATE_BASELINE and not RESUME:
    !python evaluate.py --model {MODEL} --data data --languages {" ".join(CHOSEN)} --limit 300 > /kaggle/working/baseline.txt
    check("Scoring the untouched model")
    print(open("/kaggle/working/baseline.txt").read())

## 8. Train on both GPUs

`torchrun` starts one process per GPU. Adafactor keeps the optimizer state small, so a
checkpoint of the 600M model takes about 2.5 GB of Kaggle's 20 GB working space. After 200
steps the log says how long the whole run should take; if that is longer than `HOURS`, the
run saves and stops in time, and the next session continues it (see the top).

In [ ]:
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
os.environ["TOKENIZERS_PARALLELISM"] = "false"
extra = "--lora --gradient-checkpointing" if LORA else ""
resume = f"--resume-from {RESUME}" if RESUME else ""
# A hard limit as well, 45 minutes after HOURS: if training ever hangs, it is stopped
# there and this version still finishes, with its newest checkpoint kept.
LIMIT = int(HOURS * 3600 + 45 * 60)
!timeout --kill-after=120 {LIMIT} torchrun --nproc_per_node {max(GPUS, 1)} train.py --model {MODEL} {extra} {resume} --data data --out {RUN} --languages {" ".join(CHOSEN)} --epochs {EPOCHS} --batch {BATCH} --accumulate {ACCUMULATE} --optim adafactor --save-steps 1000 --keep 2 --hours {HOURS}
if _exit_code in (124, 137):
    print(
        "Training was stopped by the hard time limit. The newest checkpoint is kept: attach "
        "this version's output to the notebook and run it again to continue from there."
    )
else:
    check("Training")

## 9. Score the fine-tuned model and compare

In [ ]:
DONE = f"{RUN}/final" if os.path.isdir(f"{RUN}/final") else f"{RUN}/partial"
if os.path.isdir(DONE):
    print("Scoring", DONE)
    !python evaluate.py --model {DONE} --data data --languages {" ".join(CHOSEN)} --limit 300 > /kaggle/working/finetuned.txt
    check("Scoring the fine-tuned model")
    print("Fine-tuned:\n" + open("/kaggle/working/finetuned.txt").read())
    if os.path.isfile("/kaggle/working/baseline.txt"):
        print("Untouched:\n" + open("/kaggle/working/baseline.txt").read())
else:
    print("No model to score in this session; the checkpoints are kept for the next one.")

Accept the model only if, for every language: chrF++ is not below the baseline on the
test sentences, it is higher on the bulletin sentences (`data/<lang>/test_domain.jsonl`,
from the forecasters' translated templates), and `numbers_kept` is 1.0 (every number, date
and amount survives). Native-speaker forecasters then review a sample before any translated
bulletin is published; machine translations stay labelled as such.

## 10. Try a bulletin sentence

In [ ]:
sentence = "Heavy rainfall (above 200 mm) is expected over western Kenya and Uganda from 3 to 9 November 2026."
for lang in CHOSEN if os.path.isdir(DONE) else []:
    print(KNOWN[lang].name, end=": ", flush=True)
    !python translate.py --model {DONE} --lang {lang} "{sentence}"

## 11. Keep the model

The run's folder is saved with this notebook version's output. To use it elsewhere, push it
to a **private** Hugging Face model (NLLB's licence, CC-BY-NC-4.0, carries over to the
fine-tuned weights: no commercial use).

In [ ]:
if PUSH_TO_HUB and os.path.isdir(f"{RUN}/final"):
    from huggingface_hub import HfApi

    api = HfApi()
    api.create_repo(PUSH_TO_HUB, private=True, exist_ok=True)
    api.upload_folder(folder_path=f"{RUN}/final", repo_id=PUSH_TO_HUB)
    print("Pushed to", PUSH_TO_HUB)
# Checkpoints are only needed to continue: drop them once the run has finished.
if os.path.isdir(f"{RUN}/final"):
    for folder in glob.glob(f"{RUN}/checkpoint-*"):
        shutil.rmtree(folder)
!du -sh /kaggle/working/*